# Practical 13 – RNNs and LSTMs

**Course:** ML6402 – Machine Learning  
**Instructor:** Dr. Ahmed Sabbah  
**University:** Al-Quds Open University


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

def load_course_csv(filename):
    candidates=[Path("data")/filename,Path("Lecture13_RNN_and_LSTM")/"data"/filename]
    local_path=next((p for p in candidates if p.exists()),None)
    url=f"https://raw.githubusercontent.com/asabbah44/ML6402-Machine-Learning-Practicals/main/Lecture13_RNN_and_LSTM/data/{filename}"
    source=local_path if local_path else url
    print("Loaded from:",source)
    return pd.read_csv(source)

train_df=load_course_csv("Diesel_Price_Train.csv")
test_df=load_course_csv("Diesel_Price_Test.csv")
train_df.head()

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras
from tensorflow.keras import layers

series=train_df["Delhi Diesel Price"].astype(float).values.reshape(-1,1)
scaler=MinMaxScaler()
scaled=scaler.fit_transform(series).ravel()

WINDOW=30
X_seq,y_seq=[],[]
for i in range(WINDOW,len(scaled)):
    X_seq.append(scaled[i-WINDOW:i])
    y_seq.append(scaled[i])

X_seq=np.array(X_seq)[...,np.newaxis]
y_seq=np.array(y_seq)

model=keras.Sequential([
    layers.Input(shape=(WINDOW,1)),
    layers.LSTM(32),
    layers.Dense(1)
])
model.compile(optimizer="adam",loss="mse")
model.fit(X_seq,y_seq,epochs=10,batch_size=32,validation_split=0.2,verbose=0)

In [ ]:
test_series=test_df["Delhi Diesel Price"].astype(float).values.reshape(-1,1)
combined=np.concatenate([series[-WINDOW:],test_series])
combined_scaled=scaler.transform(combined).ravel()

X_test_seq=[]
for i in range(WINDOW,len(combined_scaled)):
    X_test_seq.append(combined_scaled[i-WINDOW:i])

X_test_seq=np.array(X_test_seq)[...,np.newaxis]
pred_scaled=model.predict(X_test_seq,verbose=0)
pred=scaler.inverse_transform(pred_scaled).ravel()

result=pd.DataFrame({"actual":test_series.ravel(),"predicted":pred})
result.head()

## Exercise
Change the sequence window from 30 to 14 or 60 and compare prediction behavior.

# Source Chapter Worked Example – Diesel Price LSTM

The author RNN chapter uses the same diesel-price train/test datasets already stored in this practical.

Its worked example differs from the shorter introductory example above in several important ways:

- **120 time steps**;
- four stacked LSTM layers;
- 50 units per LSTM layer;
- dropout of 0.2 after each LSTM layer;
- Adam optimizer;
- mean-squared-error loss;
- inverse transformation before plotting predicted prices.

The following section reproduces that chapter structure.

In [ ]:
# Reuse train_df and test_df loaded earlier in this notebook.

from sklearn.preprocessing import MinMaxScaler
import numpy as np

source_train = train_df[["Delhi Diesel Price"]].astype(float).values

source_scaler = MinMaxScaler(feature_range=(0, 1))
source_train_scaled = source_scaler.fit_transform(source_train)

SOURCE_WINDOW = 120

X_source, y_source = [], []

for i in range(SOURCE_WINDOW, len(source_train_scaled)):
    X_source.append(source_train_scaled[i-SOURCE_WINDOW:i, 0])
    y_source.append(source_train_scaled[i, 0])

X_source = np.array(X_source)
y_source = np.array(y_source)

X_source = X_source.reshape(
    X_source.shape[0],
    X_source.shape[1],
    1
)

print("Source-chapter training shape:", X_source.shape)

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

source_lstm = keras.Sequential([
    layers.Input(shape=(SOURCE_WINDOW, 1)),

    layers.LSTM(50, return_sequences=True),
    layers.Dropout(0.2),

    layers.LSTM(50, return_sequences=True),
    layers.Dropout(0.2),

    layers.LSTM(50, return_sequences=True),
    layers.Dropout(0.2),

    layers.LSTM(50, return_sequences=False),
    layers.Dropout(0.2),

    layers.Dense(1)
])

source_lstm.compile(
    optimizer="adam",
    loss="mean_squared_error"
)

source_lstm.summary()

In [ ]:
# The author chapter uses 100 epochs.
# We use 20 by default for a faster classroom run.
source_lstm.fit(
    X_source,
    y_source,
    epochs=20,
    batch_size=32,
    verbose=0
)

In [ ]:
source_test = test_df[["Delhi Diesel Price"]].astype(float).values

combined_prices = pd.concat(
    [
        train_df["Delhi Diesel Price"],
        test_df["Delhi Diesel Price"]
    ],
    axis=0
).astype(float).values

source_inputs = combined_prices[
    len(combined_prices) - len(source_test) - SOURCE_WINDOW:
].reshape(-1, 1)

source_inputs = source_scaler.transform(source_inputs)

X_source_test = []

for i in range(SOURCE_WINDOW, len(source_inputs)):
    X_source_test.append(
        source_inputs[i-SOURCE_WINDOW:i, 0]
    )

X_source_test = np.array(X_source_test).reshape(
    -1,
    SOURCE_WINDOW,
    1
)

source_pred_scaled = source_lstm.predict(
    X_source_test,
    verbose=0
)

source_predictions = source_scaler.inverse_transform(
    source_pred_scaled
)

print("Predictions shape:", source_predictions.shape)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(source_test, label="Real Diesel Price")
plt.plot(source_predictions, label="Predicted Diesel Price")
plt.title("Diesel Price Prediction – 120 Time Steps")
plt.xlabel("Time")
plt.ylabel("Diesel Price")
plt.legend()
plt.show()

## Source-example note

The original chapter trains the stacked LSTM for **100 epochs**. To keep classroom execution time reasonable, this notebook defaults to 20 epochs. Set `epochs=100` to reproduce the source training length exactly.